# 01. Profile the raw files

Before cleaning anything, look at what the city actually published: seven yearly files,
2019-2025, from the Denver Open Data Catalog. This notebook reads them exactly as downloaded
and answers three questions: are the files consistent with each other, which columns carry
signal, and what has to be fixed.

In [1]:
import json, re
import pandas as pd
from denver311 import RAW_DIR
from denver311.clean import read_year

manifest = pd.DataFrame(json.loads((RAW_DIR / "manifest.json").read_text())).T
manifest[["kind", "item_id", "bytes", "text_lines", "downloaded_at"]]

,kind,item_id,bytes,text_lines,downloaded_at
2019,feature_service,acc3b988031749038b54c2fb8eedf320,83824288,432485,2026-09-29T14:12:40+00:00
2020,feature_service,63678a1854b740fba4ccdb34f5857538,81258103,424386,2026-09-29T14:12:50+00:00
2021,feature_service,7b1e415350424ed495b813874b02e88e,67342016,352649,2026-09-29T14:13:01+00:00
2022,feature_service,ef0a786fce784065918854d5b7eb28a7,80642821,421015,2026-09-29T14:13:07+00:00
2023,feature_service,ff6c84d8881647ca97b439971ad20a72,90510430,474701,2026-09-29T14:13:18+00:00
2024,csv_item,79f43c95bf9a4ac19ef8217c5367e6c4,97618615,455217,2026-09-29T14:12:15+00:00
2025,csv_item,5bdb9b3033d74782a4c410e548887ede,97490714,451946,2026-09-29T14:13:20+00:00
neighborhoods,feature_service_geojson,9bd8a0dd46714e3a96c489b0f8dc3edb,423032,NaN,2026-09-29T14:17:00+00:00


## Two publishing formats

2019-2023 are ArcGIS Feature Service exports; 2024-2025 are CSV uploads. The first bytes show
the difference: a UTF-8 byte-order mark and unquoted fields in the older files, fully quoted
fields in the newer ones.

In [2]:
for year in (2019, 2024):
    head = (RAW_DIR / f"311_service_requests_{year}.csv").read_bytes()[:120]
    print(year, head)

2019 b'\xef\xbb\xbfCase Summary,Case Status,Case Source,Case Created Date,Case Created dttm,Case Closed Date,Case Closed dttm,First Call'
2024 b'"Case Summary","Case Status","Case Source","Case Created Date","Case Created dttm","Case Closed Date","Case Closed dttm"'


In [3]:
def encoding_report(year):
    raw = (RAW_DIR / f"311_service_requests_{year}.csv").read_bytes()
    try:
        raw.decode("utf-8")
        enc = "utf-8" + (" with BOM" if raw[:3] == b"\xef\xbb\xbf" else "")
    except UnicodeDecodeError as e:
        enc = f"not utf-8 (byte {raw[e.start]:#x} at {e.start:,}): Windows-1252"
    return {"year": year, "encoding": enc, "crlf_lines": raw.count(b"\r\n"), "lf_only_breaks": raw.count(b"\n") - raw.count(b"\r\n")}

pd.DataFrame([encoding_report(y) for y in range(2019, 2026)])

,year,encoding,crlf_lines,lf_only_breaks
0,2019,utf-8 with BOM,432471,14
1,2020,utf-8 with BOM,424372,14
2,2021,utf-8 with BOM,352645,4
3,2022,utf-8 with BOM,421009,6
4,2023,utf-8 with BOM,474696,5
5,2024,"not utf-8 (byte 0xa0 at 10,823): Windows-1252",455214,2
6,2025,"not utf-8 (byte 0x92 at 8,557): Windows-1252",451943,2


The `lf_only_breaks` column counts line breaks inside quoted text fields. That is why a raw
line count overstates rows, and why the files are parsed with a real CSV reader, not split on
newlines. `read_year` in `src/denver311/clean.py` picks the encoding per year and normalizes
the headers (`Case Summary` and `Case_Summary` both become `case_summary`).

In [4]:
raw = pd.concat([read_year(y) for y in range(2019, 2026)], ignore_index=True)
print(f"{len(raw):,} rows, {raw.shape[1]} columns")
raw.groupby("file_year").size().rename("rows").to_frame().T

3,012,345 rows, 26 columns


file_year,2019,2020,2021,2022,2023,2024,2025
rows,432470,424371,352644,421008,474695,455214,451943


## Which columns carry signal?

In [5]:
def filled(s):
    s = s.fillna("").str.strip()
    return ((s != "") & ~s.isin(["0", "00000", "0.0"])).mean()

cols = [c for c in raw.columns if c not in ("request_id", "file_year")]
fill = raw.groupby("file_year")[cols].agg(filled).T.round(3)
fill.style.format("{:.0%}").background_gradient(cmap="Blues", axis=None)

file_year,2019,2020,2021,2022,2023,2024,2025
case_summary,100%,100%,100%,100%,100%,100%,100%
case_status,100%,100%,100%,100%,100%,100%,100%
case_source,100%,100%,100%,100%,100%,100%,100%
case_created_date,100%,100%,100%,100%,100%,100%,100%
case_created_dttm,100%,100%,100%,100%,100%,100%,100%
case_closed_date,96%,99%,96%,96%,94%,92%,95%
case_closed_dttm,96%,99%,96%,96%,94%,92%,95%
first_call_resolution,100%,100%,100%,100%,100%,100%,100%
customer_zip_code,19%,63%,55%,48%,45%,51%,52%
incident_address_1,36%,31%,26%,29%,28%,25%,25%


Four findings shape the rest of the project:

- **`Division`, `Major Area`, `Type`, `Topic`, and `Neighborhood` are empty in every year.** The
  only request-type field is the free-text `Case Summary`, and neighborhood has to be derived
  from coordinates.
- **Only about a quarter to a third of rows have a location.** Most 311 contacts are phone
  questions, not field work.
- **`Case Closed dttm` is present on 92-99% of rows,** so time to close is measurable, but only
  as the life of the 311 case record.
- **`Customer Zip Code` jumps from 19% filled in 2019 to 63% in 2020,** a capture change, not
  a population change.

In [6]:
print("First Call Resolution flag:")
print(raw["first_call_resolution"].value_counts(normalize=True).round(4))

First Call Resolution flag:
first_call_resolution
Y    0.989
N    0.011
     0.000
Name: proportion, dtype: float64


`First Call Resolution` is `Y` on 99% of rows, including cases that were routed to an agency and
closed weeks later. It is not usable as a resolution measure, so the analysis uses closure time
instead.

In [7]:
status = raw["case_status"].str.strip().value_counts()
status[status.index.str.contains("Transfer|Duplicate|Spam|Approved", regex=True)]

case_status
Closed - Transferred to External Agency    667385
Closed - Approved                          156784
Closed - Duplicate Case                     31375
Closed - Spam                               18750
Closed - Transfered to External Agency      17825
Closed - Duplicate                          14001
Closed - Not Approved                        2008
Spam                                          994
Approved                                      722
Closed - Answer Provided to Duplicate          46
Closed - Withdrawn/Duplicate                    1
Name: count, dtype: int64

Status labels include a typo (`Transfered`) and synonyms that have to be merged.

In [8]:
samples = raw.groupby("file_year")[["case_created_dttm", "case_closed_dttm"]].first()
samples

,case_created_dttm,case_closed_dttm
file_year,,
2019,1/1/2019 9:51:11 AM,01/02/2019 03:34:06 PM
2020,12/31/2020 3:26:47 PM,1/4/2021 3:47:44 PM
2021,12/30/2021 1:55:31 PM,3/30/2022 10:49:30 AM
2022,12/31/2022 2:40:59 PM,1/3/2023 7:07:48 AM
2023,12/31/2023 2:50:12 PM,1/2/2024 6:55:27 AM
2024,12/31/2024 2:04:48 PM,12/31/2024 2:04:48 PM
2025,12/31/2025 8:50:11 AM,12/31/2025 8:50:11 AM


In [9]:
lat = pd.to_numeric(raw["latitude"], errors="coerce")
lon = pd.to_numeric(raw["longitude"], errors="coerce")
copied = raw[(lat == lon) & (lat != 0)]
print(f"{len(copied):,} rows where longitude holds the latitude value")
copied[["file_year", "case_summary", "latitude", "longitude"]].head()

21,421 rows where longitude holds the latitude value


,file_year,case_summary,latitude,longitude
1745913,2023,311,39.73989051,39.73989051
1749187,2023,City & County of Denver,39.73989051,39.73989051
1749266,2023,311,39.73989051,39.73989051
1751369,2023,Encampment Reporting,39.74259462,39.74259462
1751379,2023,Encampment Reporting,39.72831862,39.72831862


In [10]:
agency = pd.crosstab(raw["agency"].str.strip(), raw["file_year"])
agency.reindex(["Environmental Health", "Public Health & Environment", "Public Works", "Safety"])

file_year,2019,2020,2021,2022,2023,2024,2025
agency,,,,,,,
Environmental Health,44141,0,0,0,0,0,0
Public Health & Environment,7461,38924,30582,28316,26591,30429,30299
Public Works,136490,95436,39795,45204,56458,53313,59827
Safety,17331,16405,39344,96138,142120,113300,134110


`Environmental Health` exists only in 2019 and `Public Health & Environment` takes over from
2020, so the two are merged. `Safety` grows from about 17,000 contacts in 2019 to over 130,000
in 2025, which becomes part of the demand story.

**Next:** `02_cleaning_decisions.ipynb` applies and documents each fix.